# Agri Term Glossary Check

Before you ship a farm advisory in eight Indian languages, you need to know which English farm terms
survive machine translation and which ones come out wrong. "Quintal" usually travels fine. "Bt cotton"
often does not. A scheme name like PM-KISAN should arguably never be translated at all: it should be
kept as-is and spelled out in the local script.

This recipe pushes 40 real Indian farm-extension terms through both Sarvam translate models into 8
Indian languages, scores every result three independent ways, and draws a term-by-language grid so you
can see at a glance where the damage is. It ends by writing a glossary file you can hand to a content
team: safe to translate, do not translate, needs a human.

Pipeline overview:
1. Translate each term from English into 8 Indian languages with `mayura:v1` and `sarvam-translate:v1`
2. Check 1 - translate the result back to English and compare it with the term you started with
3. Check 2 - ask the Language ID API what language and script the result is actually in
4. Check 3 - ask `sarvam-105b` whether a farmer would understand the same thing
5. Combine the three signals into one status per term, per language, per model
6. Draw the failure grid with matplotlib and save it to `outputs/`
7. Write a reusable glossary to `outputs/`, split into safe, do-not-translate and needs-review

Why three checks and not one: each check is blind in a different way. A round trip can come back clean
even when the intermediate text is nonsense, language ID cannot tell a real translation apart from an
English word spelled in Devanagari, and an LLM judge sounds equally confident whether or not it is
right. Where all three agree you can trust the answer. Where they disagree, the disagreement is the
interesting part, and this recipe uses it rather than hiding it.

In [ ]:
%pip install -r requirements.txt

## Setup

The API key is read from the environment or a local `.env` file. Nothing is hardcoded.

In [ ]:
from __future__ import annotations

import difflib
import json
import os
import re
import threading
import time
from collections.abc import Callable
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from sarvamai import SarvamAI
from sarvamai.core.api_error import ApiError

load_dotenv()

SARVAM_API_KEY = os.getenv("SARVAM_API_KEY")
if not SARVAM_API_KEY:
    raise RuntimeError(
        "Set SARVAM_API_KEY in your environment or .env file before running."
    )

client = SarvamAI(api_subscription_key=SARVAM_API_KEY)

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

## The glossary

Forty terms an Indian farm-extension worker uses in a normal week, across six categories. They are
deliberately chosen to be hard in different ways:

- **Seed varieties** carry release codes and institute names (`HD 2967`, `Pusa Basmati 1121`) that a
  translator may try to helpfully convert.
- **Fertilisers** are chemical names that have a settled local word in some languages and none at all
  in others.
- **Pests and diseases** are the classic trap. A literal rendering of "fall armyworm" gives a farmer
  nothing to act on.
- **Scheme names** are proper nouns. Translating "Kisan Credit Card" is a bug, not a feature.
- **Units** are the sharpest test, because getting `quintal` or `bigha` wrong changes a price or a land
  record.
- **Practices** are ordinary phrases that should translate cleanly, and act as a control group.

Each term carries a one-line plain-English gloss. The gloss is what the judge in check 3 grades
against, so it is comparing the translation with a definition rather than with its own guess at what
the term means.

In [ ]:
@dataclass(frozen=True)
class Term:
    """One English farm term, its category, and a plain-English definition."""

    text: str
    category: str
    gloss: str


TERMS: list[Term] = [
    Term("Pusa Basmati 1121", "seed", "a long-grain basmati rice variety released by IARI Pusa"),
    Term("Bt cotton", "seed", "cotton carrying a Bacillus thuringiensis gene for bollworm resistance"),
    Term("Sonalika wheat", "seed", "an early-maturing wheat variety grown across north India"),
    Term("IR-64 paddy", "seed", "a medium-duration rice variety originally bred at IRRI"),
    Term("Arka Rakshak tomato", "seed", "a disease-resistant tomato hybrid from IIHR Bengaluru"),
    Term("HD 2967 wheat", "seed", "a high-yielding irrigated wheat variety for the north Indian plains"),
    Term("Kufri Jyoti potato", "seed", "a widely grown potato variety released by CPRI Shimla"),
    Term("urea", "input", "a nitrogen fertiliser containing about 46 percent nitrogen"),
    Term("DAP", "input", "di-ammonium phosphate, a fertiliser supplying phosphorus and nitrogen"),
    Term("muriate of potash", "input", "potassium chloride, the common potassic fertiliser"),
    Term("single super phosphate", "input", "a phosphatic fertiliser that also supplies sulphur"),
    Term("zinc sulphate", "input", "a micronutrient fertiliser used on zinc-deficient soils"),
    Term("neem-coated urea", "input", "urea coated with neem oil so nitrogen is released more slowly"),
    Term("vermicompost", "input", "organic compost produced by earthworms"),
    Term("NPK 19:19:19", "input", "a water-soluble complex fertiliser with equal nitrogen, phosphorus and potassium"),
    Term("pink bollworm", "pest", "a moth larva that bores into cotton bolls and destroys the lint"),
    Term("fall armyworm", "pest", "a caterpillar that feeds inside maize whorls"),
    Term("brown planthopper", "pest", "a sap-sucking insect that causes hopper burn in paddy"),
    Term("stem borer", "pest", "a caterpillar that tunnels into paddy or sugarcane stems"),
    Term("leaf curl virus", "pest", "a whitefly-borne virus that curls chilli and cotton leaves"),
    Term("powdery mildew", "pest", "a fungal disease that leaves a white powder on leaves"),
    Term("root-knot nematode", "pest", "a soil-dwelling worm that forms galls on plant roots"),
    Term("wheat rust", "pest", "a fungal disease showing orange-brown pustules on wheat leaves and stems"),
    Term("PM-KISAN", "scheme", "a central income support scheme paying eligible farmers 6000 rupees a year"),
    Term("Pradhan Mantri Fasal Bima Yojana", "scheme", "India's national crop insurance scheme"),
    Term("Kisan Credit Card", "scheme", "a short-term crop loan facility issued to farmers as a card"),
    Term("Soil Health Card", "scheme", "a government card reporting a field's soil nutrient status"),
    Term("eNAM", "scheme", "the national online trading platform for agricultural produce markets"),
    Term("Minimum Support Price", "scheme", "the floor price the government announces for a crop"),
    Term("Rashtriya Krishi Vikas Yojana", "scheme", "a state-led agriculture development funding scheme"),
    Term("quintal", "unit", "a weight of 100 kilograms, the standard unit for crop sales"),
    Term("acre", "unit", "a land area of about 0.405 hectare"),
    Term("bigha", "unit", "a regional land area unit whose size varies from state to state"),
    Term("guntha", "unit", "a land area of about 101 square metres used in Maharashtra and Karnataka"),
    Term("hectare", "unit", "a land area of 10000 square metres"),
    Term("maund", "unit", "an older weight unit of roughly 37 kilograms, still quoted in some markets"),
    Term("drip irrigation", "practice", "watering crops drop by drop through pipes at the root zone"),
    Term("kharif season", "practice", "the monsoon cropping season, roughly June to October"),
    Term("rabi season", "practice", "the winter cropping season, roughly November to April"),
    Term("seed treatment", "practice", "coating seed with fungicide or biofertiliser before sowing"),
]

CATEGORIES = sorted({term.category for term in TERMS})
print(f"{len(TERMS)} terms across {len(CATEGORIES)} categories: {', '.join(CATEGORIES)}")

## Languages and models

Eight languages picked for farm reach: the four southern languages plus Hindi, Marathi, Bengali and
Gujarati. Codes are BCP-47 with the `-IN` suffix, which is what every Sarvam API expects.

Both translate models run on every term. They are not interchangeable. `mayura:v1` is the
general-purpose translation model and `sarvam-translate:v1` is the newer one, and a term that is safe
on one is not automatically safe on the other. Running both is the point: the grid tells you which
model to use for which category.

In [ ]:
LANGUAGES: dict[str, str] = {
    "hi-IN": "Hindi",
    "mr-IN": "Marathi",
    "bn-IN": "Bengali",
    "gu-IN": "Gujarati",
    "ta-IN": "Tamil",
    "te-IN": "Telugu",
    "kn-IN": "Kannada",
    "ml-IN": "Malayalam",
}

TRANSLATE_MODELS: list[str] = ["mayura:v1", "sarvam-translate:v1"]

## Run size, cost and caching

This is a benchmark, so it makes a lot of calls. Every cell of the grid costs four requests: one
forward translation, one round trip, one language ID, one judge call. The full sweep is:

`40 terms x 8 languages x 2 models x 4 calls = 2560 requests`

Two things keep that manageable.

**Start small.** Set `RUN_TERMS` and `RUN_LANGUAGES` to a slice first, confirm the output looks sane,
then widen to the full list.

**Everything is cached to disk.** `outputs/api_cache.json` is keyed on the exact request, so a re-run
costs nothing for cells you have already scored and an interrupted run resumes where it stopped. Delete
that file to force a genuinely fresh sweep.

`MAX_WORKERS` controls concurrency. Raise it if your plan allows it, drop it to 1 if you start seeing
429s. The retry helper below backs off, but it is cheaper not to get rate limited in the first place.

In [ ]:
# Use a slice such as TERMS[:4] and list(LANGUAGES)[:2] for a cheap smoke test first,
# then widen to the full sweep.
RUN_TERMS: list[Term] = TERMS
RUN_LANGUAGES: list[str] = list(LANGUAGES)

MAX_WORKERS = 4
ROUND_TRIP_THRESHOLD = 0.60

CACHE_PATH = OUTPUT_DIR / "api_cache.json"
SCORES_PATH = OUTPUT_DIR / "agri_term_scores.json"

planned_cells = len(RUN_TERMS) * len(RUN_LANGUAGES) * len(TRANSLATE_MODELS)
print(f"Planned grid cells: {planned_cells}")
print(f"Worst case API calls on an empty cache: {planned_cells * 4}")
print(f"Cache file: {CACHE_PATH}")

## A cached, retrying call wrapper

Every API helper below goes through `cached_call`, keyed on a string describing the exact request.
`with_retry` retries only on rate limits and server errors. A 400 means the request itself is wrong, and
retrying it just burns quota.

In [ ]:
_CACHE_LOCK = threading.Lock()


def _load_cache() -> dict[str, Any]:
    """Read the on-disk response cache, tolerating a corrupted file."""
    if CACHE_PATH.exists():
        try:
            return json.loads(CACHE_PATH.read_text(encoding="utf-8"))
        except json.JSONDecodeError:
            print(f"Cache at {CACHE_PATH} is not valid JSON. Starting from an empty cache.")
    return {}


_CACHE: dict[str, Any] = _load_cache()
print(f"Loaded {len(_CACHE)} cached responses.")


def save_cache() -> None:
    """Write the in-memory cache to disk."""
    with _CACHE_LOCK:
        snapshot = dict(_CACHE)
    CACHE_PATH.write_text(
        json.dumps(snapshot, ensure_ascii=False, indent=2), encoding="utf-8"
    )


def cached_call(key: str, fn: Callable[[], Any]) -> Any:
    """Return the cached value for key, or call fn() once and cache the result."""
    with _CACHE_LOCK:
        if key in _CACHE:
            return _CACHE[key]
    value = fn()
    with _CACHE_LOCK:
        _CACHE[key] = value
    return value


RETRYABLE_STATUS = {408, 429, 500, 502, 503, 504}


def with_retry(fn: Callable[[], Any], attempts: int = 4, base_delay: float = 2.0) -> Any:
    """Call fn(), backing off and retrying only on rate limits and server errors."""
    for attempt in range(attempts):
        try:
            return fn()
        except ApiError as exc:
            retryable = exc.status_code is None or exc.status_code in RETRYABLE_STATUS
            if not retryable or attempt == attempts - 1:
                raise
            time.sleep(base_delay * (2 ** attempt))
    raise RuntimeError("with_retry exhausted every attempt without returning a value.")

## Step 1 - translate the term

`mode="formal"` is the right register for a glossary. A colloquial mode gives you a chattier rendering
that is harder to check for exactness. `output_script="fully-native"` forces the native script, which
matters for check 2: without it you cannot tell a translation apart from an English word left sitting
in Latin letters.

In [ ]:
def translate_term(text: str, target_language_code: str, model: str) -> str:
    """Translate one English term into one Indian language with one translate model."""

    def call() -> str:
        response = client.text.translate(
            input=text,
            source_language_code="en-IN",
            target_language_code=target_language_code,
            speaker_gender="Male",
            mode="formal",
            model=model,
            output_script="fully-native",
            numerals_format="international",
        )
        return response.translated_text

    return cached_call(
        f"fwd|{model}|{target_language_code}|{text}", lambda: with_retry(call)
    )

## Check 1 - the round trip

Translate the result back into English with the same model and compare it with the term you started
with. The comparison is `difflib` character similarity combined with word overlap, taking whichever is
kinder, so that "DAP" against "di-ammonium phosphate" is not punished purely on spelling.

What this catches: meaning that fell off a cliff. What it misses: a translation that is wrong in the
target language but wrong in a way the same model reverses consistently. That blind spot is exactly why
it is one vote of three rather than the verdict.

In [ ]:
_NON_WORD_RE = re.compile(r"[^\w\s]", re.UNICODE)


def normalise(text: str) -> str:
    """Lowercase, strip punctuation, and collapse whitespace."""
    return " ".join(_NON_WORD_RE.sub(" ", text.lower()).split())


def similarity(first: str, second: str) -> float:
    """Return a 0-1 similarity for two English strings, ignoring case and punctuation."""
    left, right = normalise(first), normalise(second)
    if not left or not right:
        return 0.0
    ratio = difflib.SequenceMatcher(None, left, right).ratio()
    words_left, words_right = set(left.split()), set(right.split())
    overlap = len(words_left & words_right) / len(words_left | words_right)
    return max(ratio, overlap)


def round_trip(text: str, source_language_code: str, model: str) -> str:
    """Translate an Indian-language string back into English with the same model."""

    def call() -> str:
        response = client.text.translate(
            input=text,
            source_language_code=source_language_code,
            target_language_code="en-IN",
            speaker_gender="Male",
            mode="formal",
            model=model,
            numerals_format="international",
        )
        return response.translated_text

    return cached_call(
        f"back|{model}|{source_language_code}|{text}", lambda: with_retry(call)
    )

## Check 2 - what language did we actually get back?

The Language ID API reports a language code and a script code, and the script is the more useful half
here. If you asked for Tamil and got `Latn` back, the model left the English term untouched. If you got
`Taml`, something was written in Tamil script, though that alone does not prove it is a Tamil *word*:
"urea" transliterated is still Tamil script.

Be honest about the limits. Language ID on a one-word or three-word string is far weaker than on a
paragraph, and it will sometimes return nothing at all. `unknown` is recorded as its own outcome rather
than quietly counted as a failure.

In [ ]:
def identify_language(text: str) -> tuple[str | None, str | None]:
    """Return (language_code, script_code) for a string. Either may be None."""

    def call() -> dict[str, str | None]:
        response = client.text.identify_language(input=text)
        return {
            "language_code": response.language_code,
            "script_code": response.script_code,
        }

    data = cached_call(f"lid|{text}", lambda: with_retry(call))
    return data.get("language_code"), data.get("script_code")


def language_id_verdict(text: str, target_language_code: str) -> dict[str, str | None]:
    """Classify a translated string by the language and script the API detects in it."""
    code, script = identify_language(text)

    if script and script.lower().startswith("latn"):
        verdict = "left_in_english"
    elif code == target_language_code:
        verdict = "target_language"
    elif code in (None, "", "unknown"):
        verdict = "unknown"
    elif code == "en-IN":
        verdict = "left_in_english"
    else:
        verdict = "other_language"

    return {"verdict": verdict, "language_code": code, "script_code": script}

## Check 3 - would a farmer understand it?

`sarvam-105b` gets the English term, its gloss, the target language and the candidate translation, and
returns one of three verdicts as JSON. `temperature=0.0` keeps the judgement stable across re-runs.
`reasoning_effort=None` matters here: `sarvam-105b` reasons by default and reasoning tokens count
against `max_tokens`, which would otherwise truncate the JSON before it closes.

The judge is the only check that can separate a genuine local word from the English word simply spelled
out in the local script. Language ID calls both of those "Tamil". The judge calls the second one
`untranslated`, and that specific disagreement is what feeds the do-not-translate list at the end.

In [ ]:
JUDGE_SYSTEM_PROMPT = (
    "You are an agricultural extension officer who works with Indian farmers and reads both "
    "English and Indian languages. You check whether a farm term has been translated in a way "
    "a farmer would actually understand. You reply with JSON and nothing else."
)

VALID_VERDICTS = {"correct", "untranslated", "wrong"}

_JSON_OBJECT_RE = re.compile(r"\{.*\}", re.DOTALL)


def build_judge_prompt(term: Term, language_name: str, candidate: str) -> str:
    """Build the user prompt asking the judge to grade one translation."""
    return (
        f"English farm term: {term.text}\n"
        f"What it means: {term.gloss}\n"
        f"Target language: {language_name}\n"
        f"Proposed {language_name} text: {candidate}\n\n"
        "Choose exactly one verdict.\n"
        '"correct" means a farmer reading the proposed text understands the same thing as the '
        "English term.\n"
        '"untranslated" means the proposed text is the English term again, either in English '
        "letters or simply spelled out in the local script, with no local word used.\n"
        '"wrong" means the proposed text means something else, or is a word-by-word rendering '
        "that loses the farming sense, or names a different product, pest, scheme or unit.\n\n"
        "Reply with JSON only, in exactly this shape:\n"
        '{"verdict": "correct", "reason": "one short sentence"}'
    )


def parse_judge_reply(raw: str) -> dict[str, str]:
    """Pull a verdict and reason out of the judge's reply, tolerating stray text."""
    match = _JSON_OBJECT_RE.search(raw or "")
    if not match:
        return {"verdict": "unparsed", "reason": (raw or "").strip()[:160]}
    try:
        data = json.loads(match.group(0))
    except json.JSONDecodeError:
        return {"verdict": "unparsed", "reason": match.group(0)[:160]}

    verdict = str(data.get("verdict", "")).strip().lower()
    if verdict not in VALID_VERDICTS:
        verdict = "unparsed"
    return {"verdict": verdict, "reason": str(data.get("reason", "")).strip()[:200]}


def judge_translation(term: Term, target_language_code: str, candidate: str) -> dict[str, str]:
    """Ask sarvam-105b whether a translation preserves the farming meaning."""
    language_name = LANGUAGES[target_language_code]
    prompt = build_judge_prompt(term, language_name, candidate)

    def call() -> str:
        response = client.chat.completions(
            model="sarvam-105b",
            messages=[
                {"role": "system", "content": JUDGE_SYSTEM_PROMPT},
                {"role": "user", "content": prompt},
            ],
            temperature=0.0,
            max_tokens=300,
            reasoning_effort=None,
            wiki_grounding=False,
        )
        return response.choices[0].message.content or ""

    raw = cached_call(
        f"judge|{target_language_code}|{term.text}|{candidate}", lambda: with_retry(call)
    )
    return parse_judge_reply(raw)

## Combine the three signals into one status

Five outcomes per grid cell:

| Status | Meaning |
| --- | --- |
| `pass` | All three checks agree the term survived. Use the translation. |
| `untranslated` | The judge says no local word was used and the round trip returns the original. The model is telling you this term should be kept in English and transliterated, not translated. |
| `weak` | Exactly one check disagrees. Usable only after a human looks at it. |
| `fail` | Two or three checks disagree. Do not ship this. |
| `error` | The request itself failed. Not a language result, and kept separate so it is never counted as one. |

`untranslated` is deliberately not a failure. For a scheme name or a brand-linked input, leaving the
English term alone is the correct behaviour, and the grid should say so rather than paint it red.

In [ ]:
def score_cell(term: Term, target_language_code: str, model: str) -> dict[str, Any]:
    """Translate one term into one language with one model and score it three ways."""
    translated = translate_term(term.text, target_language_code, model)
    back = round_trip(translated, target_language_code, model)

    round_trip_score = similarity(term.text, back)
    round_trip_ok = round_trip_score >= ROUND_TRIP_THRESHOLD

    detected = language_id_verdict(translated, target_language_code)
    language_id_ok = detected["verdict"] == "target_language"

    verdict = judge_translation(term, target_language_code, translated)
    judge_ok = verdict["verdict"] == "correct"

    if verdict["verdict"] == "untranslated" and round_trip_ok:
        status = "untranslated"
    else:
        disagreements = 3 - sum([round_trip_ok, language_id_ok, judge_ok])
        status = {0: "pass", 1: "weak"}.get(disagreements, "fail")

    return {
        "term": term.text,
        "category": term.category,
        "language_code": target_language_code,
        "language": LANGUAGES[target_language_code],
        "model": model,
        "translated": translated,
        "back_translated": back,
        "round_trip_score": round(round_trip_score, 3),
        "round_trip_ok": round_trip_ok,
        "detected_language": detected["language_code"],
        "detected_script": detected["script_code"],
        "language_id_verdict": detected["verdict"],
        "language_id_ok": language_id_ok,
        "judge_verdict": verdict["verdict"],
        "judge_reason": verdict["reason"],
        "judge_ok": judge_ok,
        "status": status,
    }

## Run the sweep

Cells are scored in a thread pool because each one is four blocking HTTP calls. A cell that raises is
recorded as `error` and the sweep carries on, since losing one cell out of hundreds should not cost you
the whole run. The cache is flushed to disk as it goes, so an interrupt is survivable.

Expect this to take a while on a cold cache.

In [ ]:
jobs = [
    (term, code, model)
    for model in TRANSLATE_MODELS
    for term in RUN_TERMS
    for code in RUN_LANGUAGES
]

results: list[dict[str, Any]] = []

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
    futures = {
        pool.submit(score_cell, term, code, model): (term, code, model)
        for term, code, model in jobs
    }
    for done, future in enumerate(as_completed(futures), start=1):
        term, code, model = futures[future]
        try:
            results.append(future.result())
        except Exception as exc:  # one bad cell must not end the sweep
            results.append({
                "term": term.text,
                "category": term.category,
                "language_code": code,
                "language": LANGUAGES[code],
                "model": model,
                "status": "error",
                "error": f"{type(exc).__name__}: {exc}",
            })
        if done % 25 == 0 or done == len(futures):
            save_cache()
            print(f"scored {done} of {len(futures)} cells")

save_cache()
SCORES_PATH.write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Wrote {len(results)} scored cells to {SCORES_PATH}")

## What the numbers say

Three views of the same results: by model, by language, and by category. Read them together. A model
that wins overall can still lose badly on one category, and that is exactly the decision this recipe
exists to inform.

In [ ]:
STATUSES = ["pass", "untranslated", "weak", "fail", "error"]


def status_counts(rows: list[dict[str, Any]]) -> dict[str, int]:
    """Count rows by status, always returning every status key."""
    counts = {status: 0 for status in STATUSES}
    for row in rows:
        counts[row["status"]] = counts.get(row["status"], 0) + 1
    return counts


def group_by(rows: list[dict[str, Any]], key: str) -> dict[str, list[dict[str, Any]]]:
    """Group rows by one field, preserving first-seen order."""
    groups: dict[str, list[dict[str, Any]]] = {}
    for row in rows:
        groups.setdefault(row[key], []).append(row)
    return groups


def print_breakdown(title: str, groups: dict[str, list[dict[str, Any]]]) -> None:
    """Print a status breakdown table for a set of named row groups."""
    label_width = max([len(title)] + [len(name) for name in groups]) + 2
    header = f"{title:<{label_width}}" + "".join(f"{s:>14}" for s in STATUSES) + f"{'pass rate':>12}"
    print(header)
    print("-" * len(header))
    for name, rows in groups.items():
        counts = status_counts(rows)
        pass_rate = 100.0 * counts["pass"] / len(rows) if rows else 0.0
        line = f"{name:<{label_width}}" + "".join(f"{counts[s]:>14}" for s in STATUSES)
        print(f"{line}{pass_rate:>11.1f}%")
    print()


print_breakdown("model", group_by(results, "model"))
print_breakdown("language", group_by(results, "language"))
print_breakdown("category", group_by(results, "category"))

## Fonts, before any plotting

Matplotlib's default font draws Indic script as empty boxes, so any chart showing native-script text
needs a font that covers that script. The helpers below detect the dominant Indic script in a string and
resolve the first font from a candidate list that is actually installed, printing an install hint
instead of silently drawing boxes.

There is a second problem no font can fix. Several Indic scripts write some vowel signs *before* the
consonant they are pronounced after, and matplotlib's text layer does not reorder them. Devanagari is
the clearest case: a correctly spelled word comes out visually scrambled. Bengali and Gujarati have the
same class of sign.

So the default label mode is `romanised`, which sends the native string through Sarvam's Transliterate
API and gets back Latin letters that render correctly everywhere. Devanagari, Bengali and Gujarati are
romanised regardless of the setting. Telugu, Tamil, Kannada and Malayalam are the safer set to try in
`native` mode, but they are not immune either, so check the figure by eye before you trust it.

In [ ]:
# Indic script ranges and the fonts that can render them. Matplotlib's default
# font shows empty boxes for all of these.
INDIC_SCRIPTS: list[tuple[str, int, int, list[str]]] = [
    ("Devanagari", 0x0900, 0x097F, ["Nirmala UI", "Noto Sans Devanagari", "Mangal", "Lohit Devanagari"]),
    ("Bengali", 0x0980, 0x09FF, ["Nirmala UI", "Noto Sans Bengali", "Vrinda", "Lohit Bengali"]),
    ("Gujarati", 0x0A80, 0x0AFF, ["Nirmala UI", "Noto Sans Gujarati", "Shruti"]),
    ("Tamil", 0x0B80, 0x0BFF, ["Nirmala UI", "Noto Sans Tamil", "Latha"]),
    ("Telugu", 0x0C00, 0x0C7F, ["Nirmala UI", "Noto Sans Telugu", "Gautami"]),
    ("Kannada", 0x0C80, 0x0CFF, ["Nirmala UI", "Noto Sans Kannada", "Tunga"]),
    ("Malayalam", 0x0D00, 0x0D7F, ["Nirmala UI", "Noto Sans Malayalam", "Kartika"]),
]

# Scripts whose pre-base vowel signs matplotlib cannot reorder. Always romanised.
REORDERING_SCRIPTS = {"Devanagari", "Bengali", "Gujarati"}

LABEL_MODE = "romanised"  # "romanised" or "native"

_FONT_WARNINGS: set[str] = set()


def detect_indic_script(text: str) -> tuple[str | None, list[str]]:
    """Return (script name, font candidates) for the dominant Indic script in text."""
    counts: dict[str, int] = {}
    for char in text:
        code_point = ord(char)
        for name, low, high, _ in INDIC_SCRIPTS:
            if low <= code_point <= high:
                counts[name] = counts.get(name, 0) + 1
                break
    if not counts:
        return None, []
    top = max(counts, key=lambda name: counts[name])
    for name, _, _, fonts in INDIC_SCRIPTS:
        if name == top:
            return name, fonts
    return None, []


def resolve_font(candidates: list[str]) -> str | None:
    """Return the first candidate font that is installed, else None."""
    available = {font.name for font in fm.fontManager.ttflist}
    for name in candidates:
        if name in available:
            return name
    return None


def romanise(text: str, source_language_code: str) -> str:
    """Transliterate an Indian-language string into Latin letters."""

    def call() -> str:
        response = client.text.transliterate(
            input=text,
            source_language_code=source_language_code,
            target_language_code="en-IN",
            numerals_format="international",
            spoken_form=False,
        )
        return response.transliterated_text

    return cached_call(
        f"rom|{source_language_code}|{text}", lambda: with_retry(call)
    )


def plot_label(text: str, language_code: str) -> tuple[str, str | None]:
    """Return (label, font name) for drawing a translated string on a chart."""
    script, candidates = detect_indic_script(text)
    if script is None:
        return text, None

    if LABEL_MODE == "romanised" or script in REORDERING_SCRIPTS:
        return romanise(text, language_code), None

    font = resolve_font(candidates)
    if font is None:
        if script not in _FONT_WARNINGS:
            _FONT_WARNINGS.add(script)
            print(
                f"No installed font covers {script}. Install one of: {', '.join(candidates)}. "
                "Falling back to romanised labels for this script."
            )
        return romanise(text, language_code), None
    return text, font

## The failure grid

One column per language, one row per term, one panel per translate model, coloured by status. Row and
column labels are English, so this figure needs no Indic font at all. The native text appears in the
second figure below, which is where the font handling actually matters.

Read it column by column to pick the languages you can ship in today, and row by row to find the terms
that need a fixed, human-approved translation before anything goes out.

In [ ]:
STATUS_COLOURS = {
    "pass": "#2e7d32",
    "untranslated": "#1565c0",
    "weak": "#f9a825",
    "fail": "#c62828",
    "error": "#9e9e9e",
}
STATUS_INDEX = {name: index for index, name in enumerate(STATUS_COLOURS)}

by_cell = {(row["model"], row["term"], row["language_code"]): row for row in results}
grid_terms = [term.text for term in RUN_TERMS]

fig, axes = plt.subplots(
    1,
    len(TRANSLATE_MODELS),
    figsize=(6.5 * len(TRANSLATE_MODELS), max(6.0, 0.32 * len(grid_terms))),
    sharey=True,
)
axes = list(axes) if len(TRANSLATE_MODELS) > 1 else [axes]
colour_map = ListedColormap(list(STATUS_COLOURS.values()))

for ax, model in zip(axes, TRANSLATE_MODELS):
    matrix = [
        [
            STATUS_INDEX.get(
                by_cell.get((model, term, code), {}).get("status", "error"),
                STATUS_INDEX["error"],
            )
            for code in RUN_LANGUAGES
        ]
        for term in grid_terms
    ]
    ax.imshow(
        matrix,
        cmap=colour_map,
        vmin=-0.5,
        vmax=len(STATUS_COLOURS) - 0.5,
        aspect="auto",
    )
    ax.set_xticks(range(len(RUN_LANGUAGES)))
    ax.set_xticklabels(
        [LANGUAGES[code] for code in RUN_LANGUAGES], rotation=45, ha="right", fontsize=9
    )
    ax.set_yticks(range(len(grid_terms)))
    ax.set_yticklabels(grid_terms, fontsize=7)
    ax.set_xticks([x - 0.5 for x in range(1, len(RUN_LANGUAGES))], minor=True)
    ax.set_yticks([y - 0.5 for y in range(1, len(grid_terms))], minor=True)
    ax.grid(which="minor", color="white", linewidth=0.6)
    ax.tick_params(which="minor", length=0)
    ax.set_title(model, fontsize=11)

fig.legend(
    handles=[Patch(facecolor=colour, label=name) for name, colour in STATUS_COLOURS.items()],
    loc="lower center",
    ncol=len(STATUS_COLOURS),
    frameon=False,
    bbox_to_anchor=(0.5, -0.015),
)
fig.suptitle("Farm term translation quality by term and language", fontsize=13)
fig.tight_layout(rect=(0.0, 0.02, 1.0, 0.97))

grid_path = OUTPUT_DIR / "agri_term_grid.png"
fig.savefig(grid_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved {grid_path}")

## What went wrong, in the actual output

The grid tells you which cells failed. This panel tells you what the model actually produced, plus the
judge's one-line reason. Labels go through `plot_label`, so Devanagari, Bengali and Gujarati come back
romanised while the southern scripts are drawn natively when a font is available.

An empty panel means nothing failed hard enough to list, which is a good result rather than a broken
cell.

In [ ]:
MAX_FAILURES_SHOWN = 18

failures = [row for row in results if row.get("status") in {"fail", "weak"}]
failures.sort(key=lambda row: (row["status"] != "fail", row.get("round_trip_score", 0.0)))
shown = failures[:MAX_FAILURES_SHOWN]

if not shown:
    print("No failing or weak cells to show.")
else:
    fig, ax = plt.subplots(figsize=(13.0, 0.5 * len(shown) + 1.4))
    ax.axis("off")
    ax.set_title(
        "Worst cells: what the model actually returned", fontsize=12, loc="left", pad=26
    )

    columns = [0.0, 0.23, 0.35, 0.56]
    headers = ["term", "language", "output", "why it was flagged"]
    for index, header in enumerate(headers):
        ax.text(
            columns[index], 1.0, header, fontsize=9, fontweight="bold", transform=ax.transAxes
        )

    for position, row in enumerate(shown):
        y = 1.0 - (position + 1) * (1.0 / (len(shown) + 1))
        label, font = plot_label(row["translated"], row["language_code"])
        reason = row.get("judge_reason") or row.get("judge_verdict", "")
        values = [row["term"][:26], row["language"], label[:30], reason[:64]]
        for index, value in enumerate(values):
            font_kwargs = {"fontname": font} if font and index == 2 else {}
            ax.text(
                columns[index],
                y,
                value,
                fontsize=8,
                color=STATUS_COLOURS[row["status"]] if index == 0 else "#222222",
                transform=ax.transAxes,
                **font_kwargs,
            )

    failures_path = OUTPUT_DIR / "agri_term_failures.png"
    fig.savefig(failures_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved {failures_path}")

## The glossary file

This is the point of the whole exercise. Each term is bucketed by how it behaved across all 8 languages
and both models:

- **safe-to-translate** - at least three quarters of its cells passed. Send it through the API.
- **do-not-translate** - half or more of its cells came back `untranslated`. Keep the English term and
  transliterate it into the local script, so a farmer can at least read it aloud, instead of inventing
  a word that does not exist.
- **needs-review** - everything else. Someone who works with the crop has to fix the wording once, and
  then it becomes a pinned term your pipeline substitutes rather than translates.

Two files are written. `agri_glossary.json` is for your pipeline to load, and `agri_glossary.md` is for
the person who has to approve the wording. The JSON keeps the winning translation per language for
every safe term, so you can pin those strings instead of paying for the same call again.

In [ ]:
SAFE_THRESHOLD = 0.75
DO_NOT_TRANSLATE_THRESHOLD = 0.50
BUCKETS = ["safe-to-translate", "do-not-translate", "needs-review"]


def classify_term(rows: list[dict[str, Any]]) -> str:
    """Bucket one term based on how its cells scored across every language and model."""
    scored = [row for row in rows if row["status"] != "error"]
    if not scored:
        return "needs-review"
    untranslated = sum(1 for row in scored if row["status"] == "untranslated")
    passed = sum(1 for row in scored if row["status"] == "pass")
    if untranslated / len(scored) >= DO_NOT_TRANSLATE_THRESHOLD:
        return "do-not-translate"
    if passed / len(scored) >= SAFE_THRESHOLD:
        return "safe-to-translate"
    return "needs-review"


def best_translations(rows: list[dict[str, Any]]) -> dict[str, dict[str, str]]:
    """Pick one accepted translation per language, preferring the model listed first."""
    best: dict[str, dict[str, str]] = {}
    for model in TRANSLATE_MODELS:
        for row in rows:
            if row["model"] != model or row["status"] != "pass":
                continue
            best.setdefault(
                row["language_code"], {"text": row["translated"], "model": model}
            )
    return best


glossary: list[dict[str, Any]] = []
for term in RUN_TERMS:
    rows = [row for row in results if row["term"] == term.text]
    glossary.append({
        "term": term.text,
        "category": term.category,
        "gloss": term.gloss,
        "recommendation": classify_term(rows),
        "status_counts": status_counts(rows),
        "translations": best_translations(rows),
    })

glossary_json_path = OUTPUT_DIR / "agri_glossary.json"
glossary_json_path.write_text(
    json.dumps(glossary, ensure_ascii=False, indent=2), encoding="utf-8"
)

lines = [
    "# Agri term glossary",
    "",
    f"Generated from {len(results)} scored cells: {len(RUN_TERMS)} terms, "
    f"{len(RUN_LANGUAGES)} languages, {len(TRANSLATE_MODELS)} models.",
    "",
]
for bucket in BUCKETS:
    entries = [entry for entry in glossary if entry["recommendation"] == bucket]
    lines.append(f"## {bucket} ({len(entries)})")
    lines.append("")
    if not entries:
        lines.append("None.")
        lines.append("")
        continue
    lines.append("| term | category | pass | untranslated | weak | fail |")
    lines.append("| --- | --- | --- | --- | --- | --- |")
    for entry in entries:
        counts = entry["status_counts"]
        lines.append(
            f"| {entry['term']} | {entry['category']} | {counts['pass']} | "
            f"{counts['untranslated']} | {counts['weak']} | {counts['fail']} |"
        )
    lines.append("")

glossary_md_path = OUTPUT_DIR / "agri_glossary.md"
glossary_md_path.write_text("\n".join(lines), encoding="utf-8")

for bucket in BUCKETS:
    count = sum(1 for entry in glossary if entry["recommendation"] == bucket)
    print(f"{bucket}: {count} terms")
print(f"Wrote {glossary_json_path} and {glossary_md_path}")

## What to do with this

- Pin the `do-not-translate` terms in your pipeline and transliterate them instead. Sarvam's
  Transliterate API is the right call there: it gives a farmer something readable in their own script
  without inventing a word that does not exist in the language.
- Pin the winning strings for `safe-to-translate` terms out of `agri_glossary.json` rather than
  re-translating them on every request. It is cheaper and, more importantly, it is stable.
- Send the `needs-review` list to someone who works with the crop. Once they fix a term it becomes a
  pinned string too, and the grid gets greener without any model change at all.
- Re-run the notebook when either translate model updates. The cache means only genuinely new work
  costs anything, and the grid gives you a before-and-after you can put in front of people.

The three checks are deliberately cheap and independent. If you want a fourth, the obvious one is to
have a native speaker grade a sample by hand and compare their verdicts with the judge's. That tells
you how much to trust check 3 on your own terms, which is the only number that really matters before
you ship advisories to farmers.